# 01 · Concurrency, parallelism, and the GIL

**Time:** about 20 minutes. **You need:** basic functions, loops, and lists.

By the end, you should be able to explain why waiting and calculating need
different approaches, and choose a first tool for each.

Run the cells from top to bottom. Before each experiment, make a prediction.
The **kernel** is the Python process running your cells. Restart it to clear
variables and run everything again. No downloads or GPU are needed.

[Course guide](README.md) · Next: [Threads and I/O](02_threads_and_io.ipynb)

## 1. Two ideas that sound alike

**Concurrency:** several tasks are in progress during the same period.
They can take turns. Think of starting rice, then chopping vegetables while the rice cooks.

**Parallelism:** several tasks do work at the same instant.
Think of two cooks chopping vegetables at once.

The **CPU** is the computer's general-purpose processor. A **CPU core** is a part
of it that can execute work. A **process** is
a running program with its own memory space. A **thread** is a path of execution
inside a process. Threads in one process share memory.

An **async task** is work that can pause at an `await` and let another task run.
We will unpack `await` in lesson 04. An **event loop** coordinates tasks: it runs
ready work and resumes tasks when their waits finish. The usual asyncio event
loop runs its tasks on one thread, so coordination alone does not use several CPU cores.

## 2. Waiting versus calculating

**I/O** means input/output: reading a file, receiving a network response, or
talking to a database. **I/O-bound** work spends much of its time waiting for
these operations. Other work can use that waiting time.

**CPU-bound** work spends much of its time calculating. A long Python loop that
transforms pixels is an example. **Bound** just means limited by that resource.

Real **machine learning (ML) pipelines** mix both. A pipeline is a sequence of
steps: read image bytes → decode pixels → transform pixels → build a batch →
copy it to a GPU → train. **Decoding** turns an encoded image file into pixel
values. A **batch** is a group of samples processed together. A **GPU** is a
processor suited to doing many calculations at once.

The slowest stage is a **bottleneck**. Speeding up a different stage may have
little effect on **throughput**, the number of items completed per second.

The next cell checks your Python version and CPU count. `getattr` looks for an
optional GIL-checking function; `None` means this Python version does not provide it.

In [1]:
import os
import platform
import sys
import time

print("Python:", platform.python_version(), platform.python_implementation())
print("Logical CPUs reported by the OS:", os.cpu_count())
gil_check = getattr(sys, "_is_gil_enabled", None)
if gil_check is not None:
    print("GIL enabled right now:", gil_check())
else:
    print("Runtime GIL check unavailable; these lessons assume GIL-enabled CPython.")

Python: 3.13.7 CPython
Logical CPUs reported by the OS: 16
GIL enabled right now: True


A physical core can appear to the operating system as more than one
**logical CPU**; these share that core's resources. The count above therefore
may exceed the number of physical cores. It also may exceed the CPUs your
program is allowed to use. It helps describe the machine; it is not a required
worker count.

## 3. Measure a small baseline

A **baseline** is the simple version you compare changes against.
`time.perf_counter()` measures elapsed time, including time spent waiting.
**Overhead** is time or memory spent managing work, such as starting workers.

A few Python details in the next cells: `[function(i) for i in range(4)]` builds
a list from four calls. `assert` raises an error if its check is false; we use it
to check results. In printed numbers, `:.3f` displays three decimal places.

**Predict:** four waits of 0.08 seconds happen one after another. About how long
should that take? The next example *simulates* I/O with sleep. It does not read files.

In [2]:
def pretend_read(item_id, delay=0.08):
    time.sleep(delay)  # Stand-in for waiting on I/O, not CPU processing.
    return f"sample-{item_id}"

start = time.perf_counter()
samples = [pretend_read(i) for i in range(4)]
elapsed = time.perf_counter() - start
print(samples)
print(f"Sequential elapsed time: {elapsed:.3f} seconds")
print(f"Throughput: {len(samples) / elapsed:.1f} samples/second")
assert samples == [f"sample-{i}" for i in range(4)]

['sample-0', 'sample-1', 'sample-2', 'sample-3']
Sequential elapsed time: 0.322 seconds
Throughput: 12.4 samples/second


Expect roughly 0.32 seconds, plus overhead. **Throughput** is how many items
finish per second. **Latency** is how long one item takes from start to finish.
Overlapping waits can improve throughput even when each individual read still takes 0.08 seconds.

This idealized timeline shows two waits. `W` means waiting; `.` means not started.
It is a diagram, not a measurement:

```text
time →       0        0.08      0.16 seconds
sequential A WWWWWWWW
sequential B ........ WWWWWWWW

overlapping A WWWWWWWW
overlapping B WWWWWWWW
```

The computer does not need to calculate continuously to wait for two responses.

Now run actual Python arithmetic. There is no sleep inside this function.
The result is just a repeatable checksum: a small number summarizing the calculations.
We use it to check that different approaches do the same work.
`i * i` squares `i`, and `% 97` takes the remainder after division by 97.

In [3]:
def cpu_work(iterations):
    total = 0
    for i in range(iterations):
        total += (i * i) % 97
    return total

start = time.perf_counter()
checksums = [cpu_work(150_000) for _ in range(4)]
elapsed = time.perf_counter() - start
print("Checksums:", checksums)
print(f"CPU baseline: {elapsed:.3f} seconds")
assert len(set(checksums)) == 1

Checksums: [7199843, 7199843, 7199843, 7199843]
CPU baseline: 0.022 seconds


Changing the iteration count changes how much CPU work we request. Changing the
sleep delay changes how long we wait. Neither benchmark represents the speed of
a real image pipeline. They isolate one idea at a time.

## 4. What the GIL does

An **interpreter** is the program that runs your Python code. **CPython** is
the standard Python implementation most people install.
On a GIL-enabled CPython build, the **Global Interpreter Lock (GIL)** allows
one thread at a time to execute Python bytecode in an interpreter. Bytecode is
the instructions CPython runs for your Python code.

Threads can take turns on a Python loop, but that does not make the loop run
across several cores at once. Blocking I/O commonly releases the GIL, which is
why threads can overlap those waits. Separate processes have separate
interpreters and can run Python calculations in parallel.
[Reference: Python threading and the GIL](https://docs.python.org/3/library/threading.html#gil-and-performance-considerations).

Two qualifications matter:

- **Native code** is compiled code, often written in C or C++, that Python can
  call. Many NumPy and PyTorch operations use it. Such operations can release the
  GIL and may use their own threads. Benchmark your actual workload. An image
  decoder implemented in native code may benefit from threads too.
- Since Python 3.13, a separate **free-threaded build** can run with the GIL
  disabled. A recent Python version alone does not tell you whether your GIL
  is enabled. The check above reports it when supported.

The GIL also does **not** protect every multi-step update to shared data.
Lesson 02 shows why you can still need your own lock.
[Reference: Python free threading](https://docs.python.org/3/howto/free-threading-python.html).

## 5. Choose a starting point

| Tool | Good first experiment | Main cost or constraint |
| --- | --- | --- |
| `threading` / `ThreadPoolExecutor` | Existing blocking file or network calls | Shared data needs care; GIL limits pure Python CPU work |
| `multiprocessing` / `ProcessPoolExecutor` | Heavy independent Python calculations | Worker startup, separate memory, and transferring data |
| `asyncio` | Many I/O operations with an async API | Blocking calls can stop all tasks on the event loop |

An **API** is the set of functions a library provides. An async API gives your
program a way to wait without holding up its event loop. Adding `async` to a
function does not make its blocking work disappear.

Begin with the simplest working loop. Add concurrency when measurement shows
that waiting or CPU work is a meaningful bottleneck.

## 6. Eight cores do not promise an eightfold speedup

Suppose 20% of a job must stay sequential. The remaining 80% can be split
perfectly across workers. Even with no startup or communication cost, the
sequential part limits the speedup. **Speedup** is the old elapsed time
divided by the new elapsed time: 2× means the job takes half as long. This
limit from sequential work is often called **Amdahl's law**.

The formula below says: keep the sequential time, then divide only the
parallel part by the worker count. This is a thought experiment, not a benchmark.

**Predict:** with eight workers, will this job become eight times faster?

In [4]:
sequential_fraction = 0.20
for workers in [1, 2, 4, 8]:
    relative_time = sequential_fraction + (1 - sequential_fraction) / workers
    ideal_speedup = 1 / relative_time
    print(f"{workers} workers: at most {ideal_speedup:.2f}x in this model")

1 workers: at most 1.00x in this model
2 workers: at most 1.67x in this model
4 workers: at most 2.50x in this model
8 workers: at most 3.33x in this model


With eight workers the model gives about 3.33×. Real workers also need time
to start and exchange data. More workers can even make a small job slower.

When comparing versions, keep the input and work the same, verify the results,
and include costs that the real job will pay. Repeat timings when making a real
decision. These short lesson timings show behavior; they are not performance guarantees.

## Try it yourself

1. Change the sequential wait example from four items to eight. Predict the time first.
   Update its result check to match.
2. Pick a first tool for each job: download 50 files using a blocking client;
   run a large pure Python loop per image; make 5,000 requests with an async client.
3. Why might adding CPU workers fail to speed up training when the GPU is already busy?

<details>
<summary>Worked answers — open after trying</summary>

1. About `8 * 0.08 = 0.64` seconds sequentially. The assertion must expect eight samples.
2. A thread pool; a process pool; asyncio with a limit on active requests and pending work.
   These are starting points to measure, not universal rules.
3. If GPU computation limits throughput, extra CPU workers only prepare work sooner.
   They can increase queued data and memory use without increasing completed training steps.

</details>

## Before moving on

Explain these out loud: concurrency, parallelism, I/O-bound, CPU-bound, and GIL.
Then describe why threads can help waiting code but often do not speed up a pure
Python loop on GIL-enabled CPython.

Next: [02 · Threads and I/O](02_threads_and_io.ipynb).

Further reference: [Python timing functions](https://docs.python.org/3/library/time.html#time.perf_counter).